# Light-based geolocation with `pangeo-fish`

Temperature, bathymetry and twilight timing for an archival tag, on a
HEALPix grid. The previous version of this notebook (latitude/longitude
light grid regridded to HEALPix, lunar chapter) is in `OLD/`.

**Before this notebook**, run once per tag (see `README.md`):

1. `raw_to_dst.ipynb`: the tag folder (`dst.csv`, `tagging_events.csv`,
   `metadata.json` with `tag_type`).
2. `prepare_tag_model.ipynb`: `model.zarr` (CMEMS) in `target_root`.
3. `variance_fast.ipynb`: `temperature_variance.zarr` in `target_root`.

**Chapters**

1. Configuration
2. Tag
3. Temperature likelihood (`likelihood_fast_gu`, depth-dependent variance)
4. HEALPix grid and emission
5. Bathymetry likelihood
6. Light likelihood
7. Movement model and tracks, with and without light

**The light likelihood** (chapter 6, `pangeo_fish.light`):

- The light settings (transform, dynamic threshold, quality filter) come
  from `tag_type` in `metadata.json` (`light_settings`).
- The maps are evaluated directly on the HEALPix cells of the emission.
- Each crossing (rising or setting) is a map of its own. It goes to the
  time step that contains its observation time, like the temperature and
  pressure records, because the position of the fish (and so its local day)
  is not known.
- The maps are smoothed on the HEALPix grid (`smooth_sigma_km`, with
  `healpix_analyse`).

See `docs/how-likelihoods-are-built.md`, section 6.

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pint_xarray import unit_registry as ureg

## 1. Configuration

In [ ]:
tag_name = "18P0430"
tag_root = "s3://gfts-ifremer/tuna/tags/formatted"
scratch_root = "s3://gfts-ifremer/tuna/run/generic"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {"endpoint_url": "https://s3.gra.perf.cloud.ovh.net"},
}

bbox = {"latitude": [20, 60], "longitude": [-50, 26]}
refinement_level = 8  # about 25 km
ellipsoid = "WGS84"
# A HEALPix cell is sea when the interpolated model ocean mask is at least this
mask_threshold = 0.25
dims = ["cells"]
chunk_time = 1

# Temperature (chapter 3): reuse diff.zarr from target_root if it exists
REUSE_DIFF = True

# Emission (chapter 4)
initial_std = 1e-3  # radians, release position
recapture_std = 1e-3  # radians, end position

# Bathymetry (chapter 5)
gebco_url = "s3://gfts-reference-data/gebco_2024_new.zarr"
max_depth_m = 1000
depth_bin_size = 16

# Light (chapter 6)
HAS_LIGHT = True
# Used only when metadata.json has no tag_type
TAG_TYPE = "wc_archival"
# Single light settings for this tag, or all of them for a tag type that is
# not in pangeo_fish.light.LIGHT_SETTINGS
LIGHT_OVERRIDES = {}
# Column of dst.csv with the light record ("light" after prepare_tag_folder)
LIGHT_COLUMN = "Light Level"
SIGMA_TIMING = 10.0  # minutes
SMOOTH_KM = 28.0  # Gaussian smoothing of each light map, km
MAX_DEPTH_DBAR = 80  # deepest record allowed in a twilight window
DETECTION_WINDOW_H = 2.5
USE_FIXED_THRESHOLD = True
N_CALIB_NIGHTS = 10
N_CALIB_NIGHTS_END = 10

# Movement model (chapter 7)
maximum_speed = ureg.Quantity(5, "km / hour")
adjustment_factor = 2
# FFTConv is fast but limited to a domain of radius 10 degrees;
# HealpixAnalyseConv (exact direct sum) has no such limit.
CONV_METHOD = "HealpixAnalyseConv"
truncate = 4.0
tolerance = 1e-4
track_modes = ["mean", "mode"]
additional_track_quantities = ["speed", "distance"]

In [ ]:
target_root = f"{scratch_root}/{tag_name}"
default_chunk_dims = {"time": chunk_time} | {d: -1 for d in dims}
print(target_root)

In [ ]:
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

## 2. Tag

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=storage_options
)
metadata = dict(tag.attrs)
metadata.setdefault("tag_type", TAG_TYPE)
events = tag["tagging_events"].ds
print(metadata)
print(f"Deployment: {time_slice.start} -> {time_slice.stop}")
tag

## 3. Temperature likelihood

For each cell and day, the likelihood of the tag temperatures given the
model temperature at the depth of each record, with a variance that depends
on depth (`temperature_variance.zarr`). Daily steps: the records of a UTC
day go to the step labelled 00:00 of that day.

In [ ]:
import fsspec

fs = (
    fsspec.filesystem("s3", **storage_options)
    if target_root.startswith("s3://")
    else fsspec.filesystem("file")
)
have_diff = REUSE_DIFF and fs.exists(f"{target_root}/diff.zarr")
print("diff.zarr found, reused" if have_diff else "computing diff.zarr")

In [ ]:
from scipy.interpolate import interp1d

from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.diff import likelihood_fast_gu
from pangeo_fish.tags import adapt_model_time, reshape_by_bins

daily_log = tag_log.assign_coords(time=tag_log["time"].dt.floor("D"))

if not have_diff:
    model = xr.open_zarr(f"{target_root}/model.zarr", storage_options=storage_options)
    start = pd.Timestamp(daily_log["time"].min().item())
    end = pd.Timestamp(daily_log["time"].max().item())
    reference_model = (
        model.sel(time=adapt_model_time(slice(start, end)))
        .sel(lat=slice(*bbox["latitude"]), lon=slice(*bbox["longitude"]))
        .pipe(
            lambda ds: ds.sel(
                depth=slice(None, (daily_log["pressure"].max() + 100).compute())
            )
        )
        .chunk({"time": chunk_time, "depth": -1})
    )
    reshaped_tag = reshape_by_bins(
        daily_log,
        dim="time",
        bins=(
            reference_model.cf.add_bounds(["time"], output_dim="bounds")
            .pipe(bounds_to_bins, bounds_dim="bounds")
            .get("time_bins")
        ),
        other_dim="obs",
    ).chunk({"time": chunk_time})

    variance = xr.open_zarr(
        f"{target_root}/temperature_variance.zarr", storage_options=storage_options
    )["temperature_variance"].compute()
    f_var = interp1d(
        variance["depth"].values,
        variance.values,
        bounds_error=False,
        fill_value=(variance.values[0], variance.values[-1]),
    )
    pressure = reshaped_tag["pressure"].compute()
    var_at_depth = xr.DataArray(
        f_var(pressure.values), dims=pressure.dims, coords=pressure.coords
    )

    diff = xr.apply_ufunc(
        likelihood_fast_gu,
        reference_model["TEMP"],
        reference_model["dynamic_depth"],
        reshaped_tag["temperature"],
        reshaped_tag["pressure"],
        var_at_depth,
        input_core_dims=[["depth"], ["depth"], ["obs"], ["obs"], ["obs"]],
        output_core_dims=[[]],
        dask="parallelized",
        output_dtypes=[reference_model["TEMP"].dtype],
    )
    diff = (
        diff.to_dataset(name="diff")
        .assign(
            H0=reference_model["H0"],
            XE=reference_model["XE"],
            ocean_mask=reference_model["H0"].notnull(),
        )
        .assign_coords(
            latitude=reference_model["latitude"], longitude=reference_model["longitude"]
        )
        .compute()
    )
    diff.to_zarr(f"{target_root}/diff.zarr", mode="w", storage_options=storage_options)

diff = xr.open_zarr(f"{target_root}/diff.zarr", storage_options=storage_options)
diff["diff"].count(["lat", "lon"]).plot()

## 4. HEALPix grid and emission

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf, regrid_dataset_hpresample

diff = diff.compute()
diff = diff.assign_coords(latitude=diff["latitude"], longitude=diff["longitude"])
# regrid_dataset_hpresample expects the model grid dimensions of compute_diff
regridded, resampling_info = regrid_dataset_hpresample(
    ds=diff.rename({"lat": "yi", "lon": "xi"}),
    refinement_level=refinement_level,
    ellipsoid=ellipsoid,
    dims=dims,
    mask_threshold=mask_threshold,
)
regridded = regridded.compute()
# diff is a likelihood: normalise each time step over the cells
regridded["diff"] = regridded["diff"] / regridded["diff"].sum("cells", skipna=True)
# batch_compute_pdf_bathy (chapter 5) reads the cells and XE from this store
regridded.to_zarr(
    f"{target_root}/diff-regridded.zarr", mode="w", storage_options=storage_options
)

emission = compute_emission_pdf(
    diff_ds=regridded,
    events_ds=events,
    differences_std=1.0,  # not used: pdf is replaced by the likelihood below
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission["pdf"] = regridded["diff"]
emission = emission.compute()
cell_ids = emission["cell_ids"]
emission.to_zarr(
    f"{target_root}/emission.zarr", mode="w", storage_options=storage_options
)
emission["pdf"].count("cells").plot()

## 5. Bathymetry likelihood

In [ ]:
from pangeo_fish.bathy import (
    batch_compute_pdf_bathy,
    compute_healpix_histogram_region_bin_size,
)

gebco = xr.open_dataset(
    gebco_url, engine="zarr", chunks={}, storage_options=storage_options
).rename({"lat": "latitude", "lon": "longitude"})
subset = gebco.sel(
    latitude=slice(bbox["latitude"][0] - 0.1, bbox["latitude"][1] + 0.1),
    longitude=slice(bbox["longitude"][0] - 0.1, bbox["longitude"][1] + 0.1),
)
histo = compute_healpix_histogram_region_bin_size(
    subset,
    nside=2**refinement_level,
    max_depth_m=max_depth_m,
    depth_bin_size=depth_bin_size,
    ellipsoid=ellipsoid,
)
histo = histo.isel(
    cells=np.flatnonzero(np.isin(histo["cell_ids"].values, cell_ids.values))
)

time_bins = (
    emission[["time"]]
    .cf.add_bounds(["time"], output_dim="bounds")
    .pipe(bounds_to_bins, bounds_dim="bounds")
    .get("time_bins")
)
reshaped_tag = reshape_by_bins(
    daily_log, dim="time", bins=time_bins, other_dim="obs"
).chunk({"time": chunk_time})
pdf_bathy = batch_compute_pdf_bathy(histo, reshaped_tag, target_root, batch_size=10000)
pdf_bathy = (pdf_bathy / pdf_bathy.sum("cells", skipna=True)).compute()
position = pd.Index(pdf_bathy["cell_ids"].values).get_indexer(cell_ids.values)
assert (position >= 0).all(), "bathymetry misses some emission cells"
pdf_bathy = (
    pdf_bathy.drop_vars("cell_ids")
    .isel(cells=position)
    .reindex(time=emission["time"].values)
)
emission["pdf_bathy"] = ("time", "cells"), pdf_bathy.transpose("time", "cells").values
emission["pdf_bathy"].count("cells").plot()

## 6. Light likelihood

### 6.1 Light record and settings

In [ ]:
if HAS_LIGHT:
    import ephem

    from pangeo_fish.light import dynamic_threshold, light_settings

    settings = light_settings(metadata, **LIGHT_OVERRIDES)
    print(settings)

    df = (
        tag["/dst"]
        .ds[[LIGHT_COLUMN, "pressure", "temperature"]]
        .to_dataframe()
        .reset_index()
    )
    df["time"] = pd.to_datetime(df["time"], utc=True)
    # one-minute records: the peak light catches the crossing
    df = (
        df.sort_values("time")
        .set_index("time")
        .resample("1min")
        .agg({LIGHT_COLUMN: "max", "pressure": "mean", "temperature": "mean"})
    )
    df["light"] = settings.transform_light(df[LIGHT_COLUMN])
    df = df.dropna(subset=["light"]).reset_index()

    # the threshold rises with the moon phase
    df["time_h"] = df["time"].dt.round("h")
    phase = {
        t: float(ephem.Moon(pd.Timestamp(t).strftime("%Y/%m/%d %H:%M:%S")).phase)
        for t in df["time_h"].unique()
    }
    df["moon_phase"] = df["time_h"].map(phase)
    df["dyn_thresh"] = dynamic_threshold(df["moon_phase"], **settings.threshold)
    print(f"{len(df):,} one-minute records")

### 6.2 Crossings, quality flags and calibration

For a Wildlife Computers tag that only transmitted its crossing times
(`tag_type = "wc_psat_light"`), the pairs come from `lightloc.csv`.

In [ ]:
if HAS_LIGHT:
    from pangeo_fish.light import (
        compute_quality_flags,
        detect_twilight_events,
        pairs_from_lightloc,
        self_calibrate_solar_threshold,
    )

    release = events.sel(event_name="release")
    end = events.sel(event_name="fish_death")
    rel_lon, rel_lat = float(release.longitude), float(release.latitude)
    end_lon, end_lat = float(end.longitude), float(end.latitude)
    if np.isnan(end_lon):
        end_lon = end_lat = None

    if metadata["tag_type"] == "wc_psat_light":
        lightloc = pd.read_csv(
            fsspec.open(
                f"{tag_root}/{tag_name}/lightloc.csv", **storage_options
            ).open(),
            parse_dates=["time"],
        )
        pairs, qdf = pairs_from_lightloc(lightloc, max_depth_dbar=MAX_DEPTH_DBAR)
    else:
        pairs = detect_twilight_events(
            df,
            rel_lon,
            rel_lat,
            window_h=DETECTION_WINDOW_H,
            use_fixed_threshold=USE_FIXED_THRESHOLD,
        )
        qdf = compute_quality_flags(
            df, pairs, max_depth=MAX_DEPTH_DBAR, **settings.quality
        )

    calib = self_calibrate_solar_threshold(
        pairs,
        df,
        n_nights=N_CALIB_NIGHTS,
        release_lon=rel_lon,
        release_lat=rel_lat,
        recapture_lon=end_lon,
        recapture_lat=end_lat,
        n_nights_end=N_CALIB_NIGHTS_END if end_lon is not None else None,
    )
    print(f"{len(pairs)} nights:", qdf["flag"].value_counts().to_dict())
    print(f"threshold elevation {calib.thresh_deg:.2f} deg at release")
    if calib.thresh_deg_end is not None:
        print(f"                    {calib.thresh_deg_end:.2f} deg at the end position")

In [ ]:
if HAS_LIGHT:
    import matplotlib.pyplot as plt

    colors = {"good": "tab:blue", "partial": "tab:orange", "bad": "tab:red"}
    rise = pd.DatetimeIndex([p[0] for p in pairs])
    sett = pd.DatetimeIndex([p[1] for p in pairs])
    day_length = (sett - rise).total_seconds() / 3600
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.scatter(rise, day_length, s=6, c=[colors[f] for f in qdf["flag"]])
    ax.set_ylabel("day length (h)")
    ax.set_title("Twilight pairs (blue good, orange partial, red bad)")

### 6.3 Light maps on the HEALPix cells

In [ ]:
if HAS_LIGHT:
    from pangeo_fish.light import align_light_to_time, solar_likelihood_cells

    maps = solar_likelihood_cells(
        pairs,
        qdf,
        cell_ids,
        calib.thresh_deg,
        sigma_timing=SIGMA_TIMING,
        smooth_sigma_km=SMOOTH_KM,
    )
    pdf_light = align_light_to_time(maps, emission["time"])
    print(
        f"{maps.sizes['time']} crossings on "
        f"{int((pdf_light.n_light > 0).sum())} of {pdf_light.sizes['time']} days"
    )
    emission["pdf_light"] = pdf_light.drop_vars(["n_light", "cell_ids"])
    emission.to_zarr(
        f"{target_root}/emission.zarr", mode="w", storage_options=storage_options
    )

In [ ]:
if HAS_LIGHT:
    emission["pdf_light"].assign_coords(cell_ids=cell_ids).healpix.plot("pdf_light")

## 7. Movement model and tracks

The same fit with all the terms, and without the light term.

In [ ]:
from pangeo_fish.helpers import (
    compute_states,
    decode_trajectories,
    normalize_pdf,
    optimize_pdf,
    plot_trajectories,
)

runs = {"no_light": ["initial", "final", "mask", "pdf", "pdf_bathy"]}
if HAS_LIGHT:
    runs["with_light"] = runs["no_light"] + ["pdf_light"]

params = {}
for name, variables in runs.items():
    fit_root = f"{target_root}/{name}"
    combined = normalize_pdf(
        ds=emission[variables], chunks=default_chunk_dims, dims=dims
    )[0]
    combined.to_zarr(
        f"{fit_root}/combined.zarr", mode="w", storage_options=storage_options
    )
    params[name], updated = optimize_pdf(
        ds=combined.compute(),
        adjustment_factor=adjustment_factor,
        truncate=truncate,
        maximum_speed=maximum_speed,
        tolerance=tolerance,
        dims=dims,
        save_parameters=True,
        storage_options=storage_options,
        target_root=fit_root,
        conv_method=CONV_METHOD,
    )
    updated.to_zarr(
        f"{fit_root}/combined.zarr", mode="w", storage_options=storage_options
    )
    states = compute_states(
        target_root=fit_root,
        storage_options=storage_options,
        chunks=default_chunk_dims,
        save=True,
    )
    decode_trajectories(
        target_root=fit_root,
        states=states,
        storage_options=storage_options,
        chunks=default_chunk_dims,
        track_modes=track_modes,
        additional_track_quantities=additional_track_quantities,
        save=True,
    )
    print(name, "sigma =", params[name]["sigma"])

In [ ]:
plot_trajectories(
    target_root=f"{target_root}/with_light" if HAS_LIGHT else f"{target_root}/no_light",
    track_modes=track_modes,
    storage_options=storage_options,
    save_html=True,
)